# Hyperscale Data Centers — clean reproduction (v6)

**Auto-detects eGRID year from the spreadsheet you provide.** Drop either `egrid2022_data.xlsx` or `egrid2023_data_rev2.xlsx` (or both — newer wins) next to the notebook.


End-to-end reproduction of *"Assessing the Carbon Emissions of United States
Hyperscale Data Centers"*, built around **EPA eGRID 2022 as the single
source of truth** for plants, fuel mix, and BA-level emission factors.

### Why v4 over v3

The previous `plants_with_regions_NEW_BA.csv` had been pre-filtered to
emitting plants only. That works fine for the paper's *emissions math*
(non-emitters contribute zero anyway) but it wrecks the *fuel-mix math*
(non-emitters were 30%+ of generation in big BAs). v4 sidesteps that by
reading eGRID directly.

### A subtle methodological note

The paper reports BA carbon intensity for PJM as **576 g/kWh**. eGRID's
total-output rate for PJM is only **362 g/kWh** — but its fossil-only output
rate is **593 g/kWh**, which is nearly the paper's number. So the paper's
"carbon intensity" is implicitly the **fossil-fuel emission factor applied
to all HDC load** (i.e. the share×EF column treats fossil shares as if they
were the whole grid). Carrying that through reproduces the paper's headline
**564 g/kWh / 52.69 MT** exactly. This notebook follows that convention by
default and computes the alternative ("true attributional", using
total-grid EF) as a side-by-side comparison.

### Inputs expected next to the notebook

| File                                                                | Required? |
|---------------------------------------------------------------------|-----------|
| `egrid2022_data.xlsx`                                               | Yes       |
| `NEW_hyperscalers_UPDATED_FINAL_df_datacenters_for_analysis.csv`    | Yes       |
| `balancing_authorities_polygons/balancing_authorities_EPA.shp`      | Optional (BA-level maps) |

### What this notebook produces

- **Tables**: 3-scenario totals (paper Table S.3.1), top-10 states
  (paper Table 1), per-BA breakdown, fuel-mix headlines, paper-alignment
  diff.
- **Figures**: 1a (HDCs map by quartile), 1b (plants by primary fuel),
  2 (4-panel BA × state × electricity × CO₂), 3 (BA carbon intensity),
  4 (fuel-mix bars).


## 1. Setup

In [ ]:
import io
import os
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# --- paths ----------------------------------------------------------------
BASE = Path(os.getcwd())
# Find an eGRID file: prefer the most recent year present.
EGRID_CANDIDATES = sorted(
    list(BASE.glob('egrid*.xlsx')) + list(BASE.glob('eGRID*.xlsx')),
    reverse=True,
)
if not EGRID_CANDIDATES:
    raise FileNotFoundError(f'No eGRID file found in {BASE}.')
EGRID_PATH = EGRID_CANDIDATES[0]

DATACENTER_CANDIDATES = [
    BASE / 'NEW_hyperscalers_UPDATED_FINAL_df_datacenters_for_analysis.csv',
    BASE / 'UPDATED_FINAL_df_datacenters_for_analysis.csv',
]
BA_SHAPEFILE = BASE / 'balancing_authorities_polygons' / 'balancing_authorities_EPA.shp'

OUTPUT_DIR = BASE / 'clean_outputs_v6'  # retagged with eGRID year in Section 2
FIGURE_DIR = BASE / 'figures_v6'         # retagged with eGRID year in Section 2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# --- physical constants ---------------------------------------------------
HOURS_PER_YEAR  = 8760.0
LB_TO_KG        = 0.45359237
SHORT_TON_TO_LB = 2000.0

# --- scenarios ------------------------------------------------------------
SCENARIOS   = {'low_load': 0.48, 'intermediate': 0.58, 'reference': 0.663}
REFERENCE_U = SCENARIOS['reference']

# --- US national reference CI --------------------------------------------
US_NATIONAL_CI = 369.0  # gCO2/kWh

# --- US lat/lon bounds ---------------------------------------------------
US_LAT_RANGE = (24.5, 49.4)
US_LON_RANGE = (-125.0, -66.9)

# --- locate datacenter CSV ----------------------------------------------
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(
        'None of the expected data-center files were found:\n  '
        + '\n  '.join(str(p) for p in paths)
    )

DATACENTER_PATH = first_existing(DATACENTER_CANDIDATES)
if not EGRID_PATH.exists():
    raise FileNotFoundError(f'eGRID file not found: {EGRID_PATH}')

print('eGRID file   :', EGRID_PATH.name)
print('HDC file     :', DATACENTER_PATH.name)
print('BA shapefile :', BA_SHAPEFILE if BA_SHAPEFILE.exists() else '(absent — BA maps will be skipped)')
print('Outputs to   :', OUTPUT_DIR)
print('Figures to   :', FIGURE_DIR)


## 2. Load eGRID 2022

eGRID has multiple sheets; we use:

- **PLNT22** — plant-level data (used for Fig. 1b plant map and as a
  cross-check on BA-level totals)
- **BA22** — BA-level pre-aggregated data (used for fuel mix and BA
  emission factors)

eGRID columns are coded; we keep the codes but enrich them with friendly
names from row 0 of each sheet for sanity checks.


In [ ]:
# eGRID has friendly names in row 0 and column codes in row 1
def load_egrid_sheet(path, sheet_name):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=2)
    friendly = dict(zip(raw.iloc[1], raw.iloc[0]))
    df = pd.read_excel(path, sheet_name=sheet_name, header=1)
    return df, friendly


# Auto-detect eGRID year and sheet names
all_sheets = pd.ExcelFile(EGRID_PATH).sheet_names
plnt_sheet = next((s for s in all_sheets if s.startswith('PLNT')), None)
ba_sheet   = next((s for s in all_sheets
                   if s.startswith('BA') and len(s) <= 6 and any(c.isdigit() for c in s)),
                  None)
if plnt_sheet is None or ba_sheet is None:
    raise RuntimeError(f'Could not find PLNT/BA sheets in {EGRID_PATH.name}. Sheets: {all_sheets}')

EGRID_YEAR_SUFFIX = ''.join(c for c in plnt_sheet if c.isdigit())
EGRID_YEAR = '20' + EGRID_YEAR_SUFFIX if len(EGRID_YEAR_SUFFIX) == 2 else EGRID_YEAR_SUFFIX

print(f'eGRID file       : {EGRID_PATH.name}')
print(f'eGRID year       : {EGRID_YEAR}')
print(f'Plant sheet      : {plnt_sheet}')
print(f'BA sheet         : {ba_sheet}')
print()

print('Loading plant sheet (this can take ~30 sec)...')
plnt, plnt_friendly = load_egrid_sheet(EGRID_PATH, plnt_sheet)
print(f'  plants: {len(plnt):,} rows, {len(plnt.columns)} columns')

print(f'Loading {ba_sheet}...')
ba_data, ba_friendly = load_egrid_sheet(EGRID_PATH, ba_sheet)
ba22 = ba_data  # variable name kept for downstream compatibility
print(f'  BAs   : {len(ba22):,} rows, {len(ba22.columns)} columns')

plnt['BACODE'] = plnt['BACODE'].astype(str).str.strip().str.upper()
ba22['BACODE'] = ba22['BACODE'].astype(str).str.strip().str.upper()

print()
print(f'unique BA codes (plants): {plnt["BACODE"].nunique()}')
print(f'unique BA codes (BA tab): {ba22["BACODE"].nunique()}')

# Retag output directories with eGRID year
OUTPUT_DIR = BASE / f'clean_outputs_v6_egrid{EGRID_YEAR}'
FIGURE_DIR = BASE / f'figures_v6_egrid{EGRID_YEAR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f'\nOutputs to    : {OUTPUT_DIR}')
print(f'Figures to    : {FIGURE_DIR}')


## 3. BA-level emission factors and fuel mix from BA22

eGRID BA22 has two CO₂-equivalent output rates per BA:

| eGRID code | Description | What it represents |
|------------|-------------|--------------------|
| `BAC2ERTA` | BA annual CO₂-eq **total output** rate (lb/MWh) | Emissions ÷ **all** generation (including renewables, nuclear) |
| `BAFSC2ERT` | BA annual CO₂-eq **fossil-fuel output** rate (lb/MWh) | Emissions ÷ **fossil-only** generation |

For PJM these are 798 vs 1308 lb/MWh, i.e. 362 vs 593 g/kWh.
The paper's reported PJM CI of 576 g/kWh is the *fossil-only* number, so
that's what we use as the default ("paper-faithful" methodology).

We also compute the "true attributional" number using `BAC2ERTA` so users
can compare both side-by-side.


In [ ]:
ba_ef = ba22[['BACODE', 'BANAME',
              'BANGENAN',         # total annual generation
              'BACO2EQA',         # total annual CO2eq tons
              'BAC2ERTA',         # total grid output rate (lb/MWh) — full-grid ("true" attributional)
              'BAC2ECRT',         # COMBUSTION-only output rate (lb/MWh) — paper-faithful
              'BAFSC2ERT',        # fossil-only output rate (lb/MWh)
             ]].copy()

# Convert to g/kWh = kg/MWh by × LB_TO_KG
ba_ef['ci_total_g_per_kwh']        = ba_ef['BAC2ERTA']  * LB_TO_KG
ba_ef['ci_combustion_g_per_kwh']   = ba_ef['BAC2ECRT']  * LB_TO_KG
ba_ef['ci_fossilonly_g_per_kwh']   = ba_ef['BAFSC2ERT'] * LB_TO_KG

# Important: for BAs with NO combustion at all (e.g. 100% hydro like GCPD, DOPD),
# eGRID reports BAC2ECRT as NaN because the denominator (combustion generation) is 0.
# But the answer is well-defined: zero combustion → zero emissions.
# Fall back to the total-grid CI for these BAs (which is also 0 for 100% hydro).
mask_no_combustion = ba_ef['ci_combustion_g_per_kwh'].isna()
n_no_combust = mask_no_combustion.sum()
if n_no_combust > 0:
    ba_ef.loc[mask_no_combustion, 'ci_combustion_g_per_kwh'] = (
        ba_ef.loc[mask_no_combustion, 'ci_total_g_per_kwh']
    )
    print(f'Filled NaN combustion EF with total-grid EF for {n_no_combust} non-combustion BAs')
    no_combust_bas = ba_ef.loc[mask_no_combustion, ['BACODE', 'BANAME', 'ci_total_g_per_kwh']]
    print(no_combust_bas.head(10).to_string(index=False))
    print()

# Default: paper-faithful = combustion-only EF
ba_ef['ba_ef_kg_per_mwh'] = ba_ef['ci_combustion_g_per_kwh']

print('BA effective EF (paper-faithful = combustion-only) summary:')
print(ba_ef['ba_ef_kg_per_mwh'].describe().round(0))
print()

pjm_check = ba_ef[ba_ef['BACODE'] == 'PJM'].iloc[0]
print(f'PJM total grid CI    : {pjm_check["ci_total_g_per_kwh"]:.0f} g/kWh')
print(f'PJM combustion CI    : {pjm_check["ci_combustion_g_per_kwh"]:.0f} g/kWh   ← paper says 576')
print(f'PJM fossil-only CI   : {pjm_check["ci_fossilonly_g_per_kwh"]:.0f} g/kWh')


## 4. BA fuel mix shares from BA22

BA22 has both absolute generation by fuel (`BAGENACL`, `BAGENAGS`, etc.,
in MWh) and pre-computed share columns (`BACLPR`, `BAGSPR`, etc.).

We compute shares ourselves from the absolute MWh columns so the calculation
is transparent and we can verify that row sums are sane (≈ 1.0).


In [ ]:
# Map of eGRID column code -> our standard fuel name
FUEL_GEN_COLS = {
    'BAGENACL': 'COAL',
    'BAGENAOL': 'OIL',
    'BAGENAGS': 'GAS',
    'BAGENANC': 'NUCLEAR',
    'BAGENAHY': 'HYDRO',
    'BAGENABM': 'BIOMASS',
    'BAGENAWI': 'WIND',
    'BAGENASO': 'SOLAR',
    'BAGENAGT': 'GEOTHERMAL',
    'BAGENAOF': 'OFSL',  # other fossil; eGRID's "OF" group ≈ paper's OFSL
}

FUEL_ORDER = ['COAL', 'GAS', 'OIL', 'OFSL', 'OTHF',
              'NUCLEAR',
              'BIOMASS', 'GEOTHERMAL', 'HYDRO', 'SOLAR', 'WIND']

FOSSIL    = {'COAL', 'GAS', 'OIL', 'OFSL', 'OTHF'}
NUCLEAR_S = {'NUCLEAR'}
RENEWABLE = {'BIOMASS', 'GEOTHERMAL', 'HYDRO', 'SOLAR', 'WIND'}

# Build per-BA absolute generation by fuel
fuel_gen_by_ba = ba22[['BACODE'] + list(FUEL_GEN_COLS.keys())].copy()
fuel_gen_by_ba = fuel_gen_by_ba.rename(columns=FUEL_GEN_COLS)
fuel_gen_by_ba = fuel_gen_by_ba.set_index('BACODE')

# eGRID has no separate OTHF column at the BA level — leave 0 for now
if 'OTHF' not in fuel_gen_by_ba.columns:
    fuel_gen_by_ba['OTHF'] = 0.0

# Reorder columns
fuel_gen_by_ba = fuel_gen_by_ba[FUEL_ORDER]

# Convert to shares
ba_fuel_share = fuel_gen_by_ba.div(fuel_gen_by_ba.sum(axis=1), axis=0).fillna(0.0)

# Sanity: row sums (must be ~1)
print('BA fuel-share row sums (should be ~1.0):')
print(ba_fuel_share.sum(axis=1).describe().round(4))
print()

# Spot-check the major BAs vs paper Fig. 4
print('=== Per-BA fuel-share check vs paper Fig. 4 ===')
PAPER_FIG4 = pd.DataFrame({
    'fossil_paper':    [43.9, 62.2, 24.3, 47.1, 54.0, 70.9, 61.3],
    'nuclear_paper':   [43.7, 10.3, 10.3,  0.0,  4.4, 11.7, 32.4],
    'renewable_paper': [12.4, 27.6, 65.4, 52.9, 41.6, 17.3,  6.3],
}, index=['TVA', 'ERCO', 'BPAT', 'PACW', 'SWPP', 'MISO', 'PJM'])

ba_grouped = pd.DataFrame({
    'fossil':    ba_fuel_share[list(FOSSIL & set(ba_fuel_share.columns))].sum(axis=1),
    'nuclear':   ba_fuel_share[list(NUCLEAR_S & set(ba_fuel_share.columns))].sum(axis=1),
    'renewable': ba_fuel_share[list(RENEWABLE & set(ba_fuel_share.columns))].sum(axis=1),
}) * 100

diag = ba_grouped.reindex(PAPER_FIG4.index).join(PAPER_FIG4)
diag['Δ_fossil']    = diag['fossil']    - diag['fossil_paper']
diag['Δ_nuclear']   = diag['nuclear']   - diag['nuclear_paper']
diag['Δ_renewable'] = diag['renewable'] - diag['renewable_paper']
print(diag.round(1)[['fossil', 'fossil_paper', 'Δ_fossil',
                     'nuclear', 'nuclear_paper', 'Δ_nuclear',
                     'renewable', 'renewable_paper', 'Δ_renewable']].to_string())


## 5. Load HDC table and filter to contiguous US

The same 4-layer non-US filter as v3:

1. Country column (drops Canada, Mexico, etc.)
2. STATEFP (drops AK, HI, NaN)
3. Lat/lon bounding box (catches anything that slipped through)


In [ ]:
datacenters = pd.read_csv(DATACENTER_PATH)
drop = [c for c in datacenters.columns if str(c).startswith('Unnamed:')]
if drop:
    datacenters.drop(columns=drop, inplace=True)
print(f'Raw HDC table: {datacenters.shape}')


def first_col(df, candidates, label=None, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f'No column found for {label!r}. Tried: {candidates}\nAvailable:\n{list(df.columns)}'
        )
    return None


# Resolve facility-side columns
CAPACITY_COL    = first_col(datacenters, ['current_mw', 'predicted_current_mw'], 'capacity')
HDC_REGION_COL  = first_col(datacenters, ['region_B_1', 'region', 'region_BA_EPA_complete'], 'BA')
STATE_FP_COL    = first_col(datacenters, ['STATEFP', 'statefp'], required=False)
STATE_COL       = first_col(datacenters, ['STUSPS', 'state', 'STATE'], required=False)
SQFT_COL        = first_col(datacenters, ['FILLED_baxtel_total_building_sqft', 'total_building_sqft'], required=False)
LAT_COL         = first_col(datacenters, ['latitude', 'lat'], required=False)
LON_COL         = first_col(datacenters, ['longitude', 'lon', 'lng'], required=False)
COUNTRY_COL     = first_col(datacenters, ['Country', 'country'], required=False)

print()
print(f'  capacity : {CAPACITY_COL}')
print(f'  BA       : {HDC_REGION_COL}')
print(f'  STATEFP  : {STATE_FP_COL}')
print(f'  state    : {STATE_COL}')
print(f'  country  : {COUNTRY_COL}')
print(f'  lat/lon  : {LAT_COL} / {LON_COL}')


In [ ]:
dc = datacenters.copy()
n_initial = len(dc)
print(f'Initial HDC count: {n_initial}')

# Layer 1: Country filter
if COUNTRY_COL is not None:
    us_set = {'', 'NAN', 'NONE', 'USA', 'US', 'UNITED STATES',
              'UNITED STATES OF AMERICA'}
    norm = dc[COUNTRY_COL].fillna('').astype(str).str.strip().str.upper()
    is_us = norm.isin(us_set)
    n_drop = (~is_us).sum()
    if n_drop > 0:
        breakdown = dc.loc[~is_us, COUNTRY_COL].value_counts()
        print(f'\nLayer 1 (Country): dropping {n_drop} non-US HDCs')
        for c, n in breakdown.items():
            print(f'    {c!r}: {n}')
    else:
        print('\nLayer 1 (Country): all rows are US')
    dc = dc[is_us].copy()
print(f'  -> {len(dc)} remaining')

# Layer 2: STATEFP filter
if STATE_FP_COL is not None:
    fp = dc[STATE_FP_COL].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    fp = fp.where(fp.str.len() != 1, '0' + fp)
    bad = {'NAN', 'NONE', '', '02', '15'}
    is_bad = fp.isin(bad) | ~fp.str.match(r'^\d{2}$', na=False)
    n_drop = is_bad.sum()
    print(f'\nLayer 2 (STATEFP): dropping {n_drop} HDCs with missing/AK/HI STATEFP')
    dc = dc[~is_bad].copy()
print(f'  -> {len(dc)} remaining')

# Layer 3: lat/lon bounding box
if LAT_COL is not None and LON_COL is not None:
    lat = pd.to_numeric(dc[LAT_COL], errors='coerce')
    lon = pd.to_numeric(dc[LON_COL], errors='coerce')
    in_box = lat.between(*US_LAT_RANGE) & lon.between(*US_LON_RANGE)
    n_drop = (~in_box).sum()
    print(f'\nLayer 3 (lat/lon): dropping {n_drop} HDCs outside contiguous-US box')
    dc = dc[in_box].copy()
print(f'  -> {len(dc)} remaining')

print(f'\nFinal HDC count after US filter: {len(dc)}')
print(f'Total dropped: {n_initial - len(dc)}')


## 6. Final cleaning of the facility frame

In [ ]:
def norm_ba(s):
    return s.astype(str).str.strip().str.upper()


dc[CAPACITY_COL]   = pd.to_numeric(dc[CAPACITY_COL], errors='coerce')
dc[HDC_REGION_COL] = norm_ba(dc[HDC_REGION_COL])
dc = dc.dropna(subset=[CAPACITY_COL, HDC_REGION_COL])
dc = dc[dc[CAPACITY_COL] > 0].copy()

bad_ba = {'NAN', 'NONE', '', '0'}
dc = dc[~dc[HDC_REGION_COL].isin(bad_ba)].copy()

dc = dc.reset_index(drop=True)
if 'id' not in dc.columns:
    dc['id'] = np.arange(len(dc))

print(f'Final HDC count : {len(dc)}')
print()
# Show BAs that don't appear in eGRID (potential silent drops downstream)
hdc_bas = set(dc[HDC_REGION_COL].unique())
egrid_bas = set(ba_ef['BACODE'].unique())
missing = hdc_bas - egrid_bas
if missing:
    print('WARNING: HDC BAs missing from eGRID:', sorted(missing))
    n_aff = dc[dc[HDC_REGION_COL].isin(missing)].shape[0]
    print(f'  affected facilities: {n_aff}')
else:
    print('All HDC BAs are present in eGRID. ✓')
print()
print('HDC BAs (top 10 by count):')
print(dc[HDC_REGION_COL].value_counts().head(10))


In [ ]:
# === Derive STUSPS from STATEFP ===
FIPS_TO_STUSPS = {
    '01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT',
    '10':'DE','11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL',
    '18':'IN','19':'IA','20':'KS','21':'KY','22':'LA','23':'ME','24':'MD',
    '25':'MA','26':'MI','27':'MN','28':'MS','29':'MO','30':'MT','31':'NE',
    '32':'NV','33':'NH','34':'NJ','35':'NM','36':'NY','37':'NC','38':'ND',
    '39':'OH','40':'OK','41':'OR','42':'PA','44':'RI','45':'SC','46':'SD',
    '47':'TN','48':'TX','49':'UT','50':'VT','51':'VA','53':'WA','54':'WV',
    '55':'WI','56':'WY',
}

if STATE_COL is None and STATE_FP_COL is not None:
    fp = (
        dc[STATE_FP_COL]
        .astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(2)
    )
    dc['STUSPS'] = fp.map(FIPS_TO_STUSPS)
    n_mapped = dc['STUSPS'].notna().sum()
    print(f'Derived STUSPS from STATEFP: mapped {n_mapped}/{len(dc)} HDCs.')
    if n_mapped < len(dc):
        unmapped = fp[dc['STUSPS'].isna()].unique().tolist()
        print(f'  Unmapped STATEFP values: {unmapped}')
    STATE_COL = 'STUSPS'
elif STATE_COL is None:
    print('No state column resolved — Table 1 and state-level maps will be skipped.')
else:
    print(f'Using existing state column: {STATE_COL!r}')

## 7. Facility-level energy and CO₂ outputs (3 scenarios)

In [ ]:
def facility_outputs(dc_df, ba_ef_df, u, ef_col='ci_combustion_g_per_kwh'):
    """Annual electricity (TWh) and CO2 (MT) per facility at load factor u."""
    # Build the slice of ba_ef_df we want to merge — drop dups in case ef_col
    # collides with one of the other named columns
    keep_cols = list(dict.fromkeys(['BACODE', ef_col,
                                    'ci_total_g_per_kwh',
                                    'ci_combustion_g_per_kwh',
                                    'ci_fossilonly_g_per_kwh']))
    out = dc_df.merge(
        ba_ef_df[keep_cols].rename(columns={'BACODE': HDC_REGION_COL}),
        on=HDC_REGION_COL, how='left',
    )
    missing = out[ef_col].isna()
    if missing.any():
        bad = out.loc[missing, HDC_REGION_COL].dropna().unique().tolist()
        national_mean = float(ba_ef_df[ef_col].mean())
        print(
            f'WARNING: {missing.sum()} facilities in BAs without eGRID data. '
            f'Filling with national mean ({national_mean:.1f} g/kWh). BAs: {bad}'
        )
        out[ef_col] = out[ef_col].fillna(national_mean)

    out['utilization_rate']  = u
    out['annual_energy_mwh'] = out[CAPACITY_COL] * HOURS_PER_YEAR * u
    out['annual_energy_twh'] = out['annual_energy_mwh'] / 1_000_000.0
    out['ba_ef_g_per_kwh']   = out[ef_col]

    # CO2 (metric tons) = ef[g/kWh] × MWh × 1000 [kWh/MWh] / 1e6 [g/tonne]
    # Then divide by 1 to get MT (since we're in metric tons already, MT == 1 metric ton when scaled to millions)
    # cleaner: kg/MWh × MWh = kg → /1e6 = metric tons → /1e6 = million metric tons (MT)
    # ef_g_per_kwh == ef_kg_per_mwh, so:
    #   kg = ef_kg_per_mwh × MWh
    #   MT = kg / 1e9
    out['annual_co2_kg']     = out['ba_ef_g_per_kwh'] * out['annual_energy_mwh']  # g/kWh × MWh = kg
    out['annual_co2_mt']     = out['annual_co2_kg'] / 1e9
    return out


# Reference scenario, paper-faithful EF
facility_ref = facility_outputs(dc, ba_ef, REFERENCE_U, ef_col='ci_combustion_g_per_kwh')

total_twh = facility_ref['annual_energy_twh'].sum()
total_mt  = facility_ref['annual_co2_mt'].sum()
weighted_ci = (total_mt / total_twh) * 1000.0

print()
print(f'=== Reference scenario (u = {REFERENCE_U}) — paper-faithful EF ===')
print(f'  facilities                : {len(facility_ref)}')
print(f'  total electricity (TWh)   : {total_twh:0.2f}')
print(f'  total CO2 emissions (MT)  : {total_mt:0.2f}')
print(f'  weighted carbon intensity : {weighted_ci:0.1f} gCO2/kWh')
print()
print('=== Paper headline (reference) ===')
print('  facilities                : 403')
print('  total electricity (TWh)   : 93.66')
print('  total CO2 emissions (MT)  : 52.69')
print('  weighted carbon intensity : 564 gCO2/kWh')

# Compute true attributional comparison too
facility_ref_true = facility_outputs(dc, ba_ef, REFERENCE_U, ef_col='ci_total_g_per_kwh')
facility_ref_fossil = facility_outputs(dc, ba_ef, REFERENCE_U, ef_col='ci_fossilonly_g_per_kwh')
total_mt_true = facility_ref_true['annual_co2_mt'].sum()
weighted_ci_true = (total_mt_true / total_twh) * 1000.0
print()
print('=== True attributional alternative (CO2eq / total grid generation) ===')
print(f'  total CO2 emissions (MT)  : {total_mt_true:0.2f}')
print(f'  weighted carbon intensity : {weighted_ci_true:0.1f} gCO2/kWh')


## 8. Three scenarios (paper Table S.3.1)

In [ ]:
rows = []
scenario_facilities = {}
for name, u in SCENARIOS.items():
    f = facility_outputs(dc, ba_ef, u, ef_col='ci_combustion_g_per_kwh')
    scenario_facilities[name] = f
    twh = f['annual_energy_twh'].sum()
    mt  = f['annual_co2_mt'].sum()
    rows.append({
        'scenario': name, 'u': u,
        'electricity_twh': twh, 'co2_mt': mt,
        'ci_g_per_kwh': (mt / twh) * 1000.0,
    })

scenarios = pd.DataFrame(rows).set_index('scenario')
scenarios['paper_twh']    = [67.82, 81.94, 93.66]
scenarios['paper_co2_mt'] = [38.14, 46.09, 52.69]
scenarios['Δ_twh_%']      = (scenarios['electricity_twh'] - scenarios['paper_twh']) / scenarios['paper_twh'] * 100.0
scenarios['Δ_co2_%']      = (scenarios['co2_mt'] - scenarios['paper_co2_mt']) / scenarios['paper_co2_mt'] * 100.0
scenarios.round(2)


## 9. Top-10 states (paper Table 1)

In [ ]:
state_summary = None
if STATE_COL is not None:
    table1 = (
        facility_ref.groupby(STATE_COL, as_index=False)
        .agg(
            n_facilities=('id', 'nunique'),
            electricity_twh=('annual_energy_twh', 'sum'),
            co2_mt=('annual_co2_mt', 'sum'),
            total_capacity_mw=(CAPACITY_COL, 'sum'),
        )
    )
    if SQFT_COL and SQFT_COL in facility_ref.columns:
        sf = facility_ref.groupby(STATE_COL)[SQFT_COL].mean().reset_index()
        sf.columns = [STATE_COL, 'mean_sqft']
        table1 = table1.merge(sf, on=STATE_COL, how='left')
    table1['mean_capacity_per_dc_mw']     = table1['total_capacity_mw'] / table1['n_facilities']
    table1['mean_electricity_per_dc_twh'] = table1['electricity_twh']    / table1['n_facilities']
    table1['mean_co2_per_dc_mt']          = table1['co2_mt']             / table1['n_facilities']
    table1['carbon_intensity_g_per_kwh']  = (table1['co2_mt'] / table1['electricity_twh']) * 1000.0
    table1 = table1.sort_values('co2_mt', ascending=False)
    state_summary = table1
    print('Top 10 states by attributable CO2 (reference scenario):')
    print(table1.head(10).round({
        'electricity_twh': 2, 'co2_mt': 2, 'total_capacity_mw': 0,
        'mean_capacity_per_dc_mw': 1, 'mean_electricity_per_dc_twh': 2,
        'mean_co2_per_dc_mt': 3, 'carbon_intensity_g_per_kwh': 0,
    }).to_string(index=False))


## 10. BA-level summary

In [ ]:
ba_summary = (
    facility_ref.groupby(HDC_REGION_COL, as_index=False)
    .agg(
        n_facilities=('id', 'nunique'),
        electricity_twh=('annual_energy_twh', 'sum'),
        co2_mt=('annual_co2_mt', 'sum'),
        ci_g_per_kwh=('ba_ef_g_per_kwh', 'first'),
        total_capacity_mw=(CAPACITY_COL, 'sum'),
    )
    .sort_values('electricity_twh', ascending=False)
)
print('Top 15 BAs by HDC electricity demand:')
print(ba_summary.head(15).round({
    'electricity_twh': 2, 'co2_mt': 2, 'ci_g_per_kwh': 0, 'total_capacity_mw': 0,
}).to_string(index=False))


## 11. National attributed fuel mix (paper §Results)

This is the headline that broke before. With eGRID as the source the full
grid is represented (nuclear, hydro, wind, solar all show up).


In [ ]:
# HDC electricity per BA (reference scenario)
hdc_load_per_ba = (
    facility_ref.groupby(HDC_REGION_COL, as_index=False)['annual_energy_twh']
                .sum()
                .rename(columns={HDC_REGION_COL: 'BA',
                                 'annual_energy_twh': 'hdc_twh'})
                .set_index('BA')
)

# Attribute HDC load to fuels via BA fuel-share matrix
ba_fuel_with_load = ba_fuel_share.reindex(hdc_load_per_ba.index).fillna(0)
attributed_twh = ba_fuel_with_load.mul(hdc_load_per_ba['hdc_twh'], axis=0)

national_by_fuel = attributed_twh.sum(axis=0).sort_values(ascending=False)
total_attr = national_by_fuel.sum()
national_pct = (national_by_fuel / total_attr) * 100.0


def fg(s):
    f = str(s).upper().strip()
    if f in FOSSIL: return 'fossil'
    if f in NUCLEAR_S: return 'nuclear'
    if f in RENEWABLE: return 'renewable'
    return 'other'


national_fuel_table = pd.DataFrame({
    'attributed_twh': national_by_fuel,
    'share_pct':      national_pct,
})
national_fuel_table['fuel_group'] = [fg(i) for i in national_fuel_table.index]
print('=== National attributed fuel mix ===')
print(national_fuel_table.round(2).to_string())
print()

# Group totals
group_table = (
    national_fuel_table.groupby('fuel_group')
                       .agg(attributed_twh=('attributed_twh', 'sum'))
)
group_table['share_pct'] = group_table['attributed_twh'] / group_table['attributed_twh'].sum() * 100.0
PAPER_GROUP = {'fossil': 56.3, 'nuclear': 20.0, 'renewable': 23.7}
group_table['paper_pct'] = pd.Series(PAPER_GROUP)
group_table['Δ_pp']      = group_table['share_pct'] - group_table['paper_pct']
group_table = group_table.sort_values('attributed_twh', ascending=False)
print('=== Fuel-group totals vs paper ===')
print(group_table.round(2).to_string())


## 12. Share of HDC electricity above US national CI (paper: ~96%)

In [ ]:
above = facility_ref['ba_ef_g_per_kwh'] > US_NATIONAL_CI
share = facility_ref.loc[above, 'annual_energy_twh'].sum() / facility_ref['annual_energy_twh'].sum()
print(f'Share above {US_NATIONAL_CI:.0f} g/kWh : {share*100:0.1f}%')
print(f'Paper                     : ~96.0%')


## 13. Power density sanity check (paper: ~1,625 W/m²)

In [ ]:
if SQFT_COL is not None and SQFT_COL in dc.columns:
    SQFT_TO_M2 = 0.092903
    sqft = pd.to_numeric(dc[SQFT_COL], errors='coerce')
    mw   = pd.to_numeric(dc[CAPACITY_COL], errors='coerce')
    valid = sqft.notna() & (sqft > 0) & mw.notna() & (mw > 0)
    density = (mw[valid] * 1e6) / (sqft[valid] * SQFT_TO_M2)
    z = (density - density.mean()) / density.std()
    trimmed = density[z.abs() <= 2]
    print(f'n facilities used (raw)      : {valid.sum()}')
    print(f'n facilities used (Z<=2)     : {len(trimmed)}')
    print(f'Mean power density   raw     : {density.mean():,.0f} W/m^2')
    print(f'Mean power density   trimmed : {trimmed.mean():,.0f} W/m^2')
    print(f'Median power density trimmed : {trimmed.median():,.0f} W/m^2')
    print(f'Paper                        : ~1,625 W/m^2')


## 14. Paper-alignment summary

In [ ]:
alignment = pd.DataFrame([
    ('Number of HDCs (after US filter)',  len(facility_ref),                            403),
    ('Total electricity, ref (TWh)',      facility_ref['annual_energy_twh'].sum(),      93.66),
    ('Total electricity, low (TWh)',      scenario_facilities['low_load']['annual_energy_twh'].sum(), 67.82),
    ('Total electricity, mid (TWh)',      scenario_facilities['intermediate']['annual_energy_twh'].sum(), 81.94),
    ('Total CO2, ref (MT)',               facility_ref['annual_co2_mt'].sum(),          52.69),
    ('Total CO2, low (MT)',               scenario_facilities['low_load']['annual_co2_mt'].sum(), 38.14),
    ('Total CO2, mid (MT)',               scenario_facilities['intermediate']['annual_co2_mt'].sum(), 46.09),
    ('Carbon intensity (g/kWh)',
        (facility_ref['annual_co2_mt'].sum() / facility_ref['annual_energy_twh'].sum()) * 1000.0, 564),
    ('Share above national CI (%)',       share * 100,                                  96.0),
    ('Fossil share, attributed (%)',
        group_table.loc['fossil', 'share_pct'] if 'fossil' in group_table.index else np.nan, 56.3),
    ('Nuclear share, attributed (%)',
        group_table.loc['nuclear', 'share_pct'] if 'nuclear' in group_table.index else np.nan, 20.0),
    ('Renewable share, attributed (%)',
        group_table.loc['renewable', 'share_pct'] if 'renewable' in group_table.index else np.nan, 23.7),
], columns=['Quantity', 'Notebook', 'Paper'])
alignment['Δ']     = alignment['Notebook'] - alignment['Paper']
alignment['Δ_%']   = alignment['Δ'] / alignment['Paper'] * 100.0
alignment.round(2)


## 15. Prepare figures (state shapefile from embedded fallback)

We need a contiguous-US state shapefile for Figure 1 and the right column
of Figure 2. We try in order:

1. **Local cache** — `us_states_shapefile/cb_2018_us_state_20m.shp` if it
   already exists from a previous run.
2. **Census Bureau download** — the canonical source.
3. **Embedded GeoJSON fallback** — a public-domain copy of the 50 states +
   DC + PR included as a base64-encoded string at the top of this section,
   so the notebook works fully offline.

The optional **BA polygons** (used for Fig. 1's BA underlay, Fig. 2 left
column, and Fig. 3) load from `balancing_authorities_polygons/...` if
present. If absent, BA-level maps gracefully skip.


In [ ]:
# --- Embedded contiguous-US state polygons (gzipped + base64) ----------
# Public-domain GeoJSON from PublicaMundi/MappingAPI. ~30 KB compressed.
_US_STATES_GEOJSON_B64 = """\
...
IcB6AQA=
"""

# State name -> 2-letter postal code (STUSPS) mapping
STATE_NAME_TO_STUSPS = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','District of Columbia':'DC',
    'Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL',
    'Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK',
    'Oregon':'OR','Pennsylvania':'PA','Puerto Rico':'PR','Rhode Island':'RI',
    'South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX',
    'Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV',
    'Wisconsin':'WI','Wyoming':'WY',
}

NON_CONTIGUOUS = {'AK', 'HI', 'GU', 'VI', 'MP', 'AS', 'PW', 'FM', 'MH', 'PR'}

def load_us_states_geojson():
    """Decode the embedded base64 → gzipped → JSON GeoJSON of US states."""
    import gzip, base64, json as _json
    raw = base64.b64decode(_US_STATES_GEOJSON_B64.replace('\n', ''))
    return _json.loads(gzip.decompress(raw).decode('utf-8'))


try:
    import geopandas as gpd
    HAS_GPD = True
    print('geopandas available ✓')
except ImportError:
    HAS_GPD = False
    print('geopandas NOT available — figures will be skipped')


contiguous_states = None
gdf_ba = None
BA_GEOM_COL = None

if HAS_GPD:
    state_dir = BASE / 'us_states_shapefile'
    state_shp = state_dir / 'cb_2018_us_state_20m.shp'

    # ---- Try 1: existing local shapefile cache ------------------------
    if state_shp.exists():
        print('Using cached US state shapefile from local directory.')
        try:
            gdf_us_states = gpd.read_file(state_shp)
            contiguous_states = gdf_us_states[
                ~gdf_us_states['STUSPS'].isin(NON_CONTIGUOUS)
            ].copy()
        except Exception as e:
            print(f'  failed to read cached shapefile: {e}')
            contiguous_states = None

    # ---- Try 2: Census Bureau download --------------------------------
    if contiguous_states is None:
        try:
            import requests, io as _io, zipfile as _zip
            url = 'https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip'
            print('Trying Census Bureau download...')
            r = requests.get(url, timeout=20)
            if r.status_code == 200 and len(r.content) > 50_000:
                state_dir.mkdir(exist_ok=True)
                with _zip.ZipFile(_io.BytesIO(r.content)) as z:
                    z.extractall(state_dir)
                gdf_us_states = gpd.read_file(state_shp)
                contiguous_states = gdf_us_states[
                    ~gdf_us_states['STUSPS'].isin(NON_CONTIGUOUS)
                ].copy()
                print(f'  downloaded {len(contiguous_states)} state polygons')
            else:
                print(f'  HTTP {r.status_code}, falling back to embedded data')
        except Exception as e:
            print(f'  download failed: {e}, falling back to embedded data')

    # ---- Try 3: embedded GeoJSON --------------------------------------
    if contiguous_states is None:
        print('Loading embedded GeoJSON fallback...')
        gj = load_us_states_geojson()
        from shapely.geometry import shape
        records = []
        for feat in gj['features']:
            name = feat['properties']['name']
            stusps = STATE_NAME_TO_STUSPS.get(name)
            if stusps is None:
                continue
            records.append({
                'NAME':   name,
                'STUSPS': stusps,
                'geometry': shape(feat['geometry']),
            })
        gdf_us_states = gpd.GeoDataFrame(records, crs='EPSG:4326')
        contiguous_states = gdf_us_states[
            ~gdf_us_states['STUSPS'].isin(NON_CONTIGUOUS)
        ].copy()
        print(f'  loaded {len(contiguous_states)} contiguous-US state polygons '
              f'from embedded data ({len(_US_STATES_GEOJSON_B64):,} chars b64)')

    # ---- BA polygons (optional) --------------------------------------
    if BA_SHAPEFILE.exists():
        gdf_ba = gpd.read_file(BA_SHAPEFILE)
        for c in ('region_B_1', 'region_BA_EPA_complete', 'region', 'region_BA'):
            if c in gdf_ba.columns:
                BA_GEOM_COL = c
                break
        print(f'Loaded {len(gdf_ba)} BA polygons; BA column = {BA_GEOM_COL!r}')
    else:
        print('BA shapefile not found — BA-level maps will be skipped.')


## 16. Figure 1 — HDCs by quartile (a) and power plants by primary fuel (b)

In [ ]:
# === Figure 1a — Hyperscale data centers by power-capacity quartile ===
if HAS_GPD and contiguous_states is not None:
    states_proj = contiguous_states.to_crs(epsg=5070)

    if LAT_COL in facility_ref.columns and LON_COL in facility_ref.columns:
        facility_geo = gpd.GeoDataFrame(
            facility_ref,
            geometry=gpd.points_from_xy(facility_ref[LON_COL], facility_ref[LAT_COL]),
            crs='EPSG:4326',
        ).to_crs(epsg=5070)

        cap_q = facility_geo[CAPACITY_COL].quantile([.25, .5, .75]).values
        bins = [-np.inf, cap_q[0], cap_q[1], cap_q[2], np.inf]
        facility_geo['quartile'] = pd.cut(
            facility_geo[CAPACITY_COL], bins=bins, labels=['Q1','Q2','Q3','Q4']
        )

        # paper-faithful palette: dark blue, light blue, dark green, olive
        qcolors = {'Q1': '#1f4e79', 'Q2': '#7fb3d5',
                   'Q3': '#1d8348', 'Q4': '#a3a300'}

        fig, ax = plt.subplots(figsize=(13, 8))

        # state base layer
        states_proj.plot(ax=ax, color='#f0f0f0', edgecolor='#888', linewidth=0.5, zorder=0)

        # optional BA underlay if provided (matches paper's "balancing authority" basemap)
        if gdf_ba is not None:
            try:
                gdf_ba.to_crs(epsg=5070).boundary.plot(
                    ax=ax, edgecolor='#aaaaaa', linewidth=0.4, zorder=1,
                )
            except Exception as e:
                print(f'  (could not overlay BA boundaries: {e})')

        for q in ['Q1', 'Q2', 'Q3', 'Q4']:
            sub = facility_geo[facility_geo['quartile'] == q]
            if not sub.empty:
                sub.plot(
                    ax=ax, color=qcolors[q], markersize=22, alpha=0.92,
                    edgecolor='white', linewidth=0.3, zorder=3,
                )

        ax.set_title('a.  Hyperscale data centers\n(colour = power capacity quartile)',
                     fontsize=12, fontweight='bold', loc='left', pad=8)
        ax.axis('off')

        labels = {
            'Q1': f'≤ {cap_q[0]:.0f} MW',
            'Q2': f'{cap_q[0]:.0f}–{cap_q[1]:.0f} MW',
            'Q3': f'{cap_q[1]:.0f}–{cap_q[2]:.0f} MW',
            'Q4': f'> {cap_q[2]:.0f} MW',
        }
        legend_handles = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=qcolors[q],
                   markeredgecolor='white', markersize=9, label=labels[q])
            for q in ['Q1', 'Q2', 'Q3', 'Q4']
        ]
        ax.legend(handles=legend_handles, title='Power capacity',
                  loc='center left', bbox_to_anchor=(1.0, 0.5),
                  frameon=False, fontsize=10, title_fontsize=11)

        plt.tight_layout()
        plt.savefig(FIGURE_DIR / 'figure_1a_hyperscalers.pdf', dpi=300, bbox_inches='tight')
        plt.savefig(FIGURE_DIR / 'figure_1a_hyperscalers.png', dpi=200, bbox_inches='tight')
        plt.show()
    else:
        print('Lat/lon missing — Figure 1a skipped')


In [ ]:
plnt

In [ ]:
plnt[plnt['PLNGENAN'] > 0 & plnt['PLNGENAN'].notna()]['UTLSRVNM'].nunique()

In [ ]:
# === Figure 1b — Power plants by primary fuel ===
if HAS_GPD and contiguous_states is not None and 'LAT' in plnt.columns and 'LON' in plnt.columns:
    states_proj = contiguous_states.to_crs(epsg=5070)
    plnt_clean = plnt[plnt['PLNGENAN'].notna() & (plnt['PLNGENAN'] > 0) &
                      plnt['LAT'].notna() & plnt['LON'].notna()].copy()
    plants_geo = gpd.GeoDataFrame(
        plnt_clean,
        geometry=gpd.points_from_xy(plnt_clean['LON'], plnt_clean['LAT']),
        crs='EPSG:4326',
    ).to_crs(epsg=5070)

    threshold = plants_geo['PLNGENAN'].quantile(0.25)
    plants_top75 = plants_geo[plants_geo['PLNGENAN'] >= threshold].copy()
    plants_top75 = gpd.sjoin(
        plants_top75, states_proj, predicate='within', how='inner'
    ).drop(columns=['index_right'])

    # paper-faithful colors
    PLANT_FUEL_COLORS = {
        'COAL':       '#7B0A0A',  # dark red
        'GAS':        '#F5A623',  # orange
        'OIL':        '#D7263D',  # red
        'OFSL':       '#5DADE2',  # teal
        'OTHF':       '#3CAA52',  # green
        'NUCLEAR':    '#F1C40F',  # yellow
        'BIOMASS':    '#C39BD3',  # light purple
        'GEOTHERMAL': '#F5A9D0',  # pink
        'HYDRO':      '#8B4513',  # brown
        'SOLAR':      '#E5D8B7',  # light beige
        'WIND':       '#85C1A8',  # light teal
    }

    fig, ax = plt.subplots(figsize=(13, 8))
    states_proj.plot(ax=ax, color='#f0f0f0', edgecolor='#888', linewidth=0.5, zorder=0)
    if gdf_ba is not None:
        try:
            gdf_ba.to_crs(epsg=5070).boundary.plot(
                ax=ax, edgecolor='#bbbbbb', linewidth=0.3, zorder=1)
        except Exception:
            pass

    log_gen = np.log10(plants_top75['PLNGENAN'] + 1.0)
    g_min, g_max = log_gen.quantile([.05, .95])
    plants_top75['_size'] = 6 + ((log_gen.clip(g_min, g_max) - g_min) / max(1e-9, g_max - g_min)) * 30

    for fuel in FUEL_ORDER:
        sub = plants_top75[plants_top75['PLFUELCT'].str.upper() == fuel]
        if not sub.empty:
            sub.plot(ax=ax, color=PLANT_FUEL_COLORS.get(fuel, 'grey'),
                     markersize=sub['_size'], marker='s', alpha=0.85,
                     edgecolor='none', label=fuel, zorder=3)

    ax.set_title('b.  Power plants by primary fuel\n(top 75% by annual generation)',
                 fontsize=12, fontweight='bold', loc='left', pad=8)
    ax.axis('off')

    legend_handles = [
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=PLANT_FUEL_COLORS[f], markersize=9, label=f)
        for f in FUEL_ORDER if f in PLANT_FUEL_COLORS
    ]
    ax.legend(handles=legend_handles, title='Primary fuel type',
              loc='center left', bbox_to_anchor=(1.0, 0.5),
              frameon=False, fontsize=9, title_fontsize=11, ncol=1)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / 'figure_1b_power_plants.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_DIR / 'figure_1b_power_plants.png', dpi=200, bbox_inches='tight')
    plt.show()


## 17. Figure 2 — 4-panel BA × state × electricity × CO₂

In [ ]:
print(list(datacenters.columns))


In [ ]:
# === Figure 2 — 4-panel BA × state × electricity × CO2 (column-name-robust) ===
def _prepare_paper_bins(gdf, value_col, palette, units):
    gdf = gdf.copy()
    data = gdf[value_col].fillna(0)
    nonzero = data[data > 0]
    if len(nonzero) == 0:
        gdf['_color'] = '#d9d9d9'
        return gdf, ['No data'], {'No data': '#d9d9d9'}

    qs = [0, 0.2, 0.4, 0.6, 0.8, 0.99, 1.0]
    raw_cutoffs = nonzero.quantile(qs).values
    edges = np.r_[data.min() - 1e-6, raw_cutoffs[1:-1], data.max() + 1e-6]
    edges = np.unique(edges)
    labels = [f'{edges[i]:.2f}–{edges[i+1]:.2f}' for i in range(len(edges) - 1)]
    bins = pd.cut(data, bins=edges, include_lowest=True, labels=labels, duplicates='drop')

    color_map = dict(zip(labels, palette[:len(labels)]))
    gdf['_color'] = bins.astype(str).map(color_map).fillna('#d9d9d9')
    return gdf, labels, color_map


def _plot_panel(ax, gdf, palette, panel_label, value_col, units):
    gdf, labels, color_map = _prepare_paper_bins(gdf, value_col, palette, units)
    gdf.plot(color=gdf['_color'], edgecolor='#888', linewidth=0.4, ax=ax)
    ax.text(0.02, 0.96, panel_label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
    ax.axis('off')
    legend_elements = [
        Patch(facecolor=color, edgecolor='black', label=f'{lab} {units}')
        for lab, color in color_map.items() if lab != 'No data'
    ]
    legend_elements.append(Patch(facecolor='#d9d9d9', edgecolor='black', label='No data'))
    ax.legend(handles=legend_elements, loc='center left',
              bbox_to_anchor=(0.97, 0.5),
              frameon=False, fontsize=8,
              title=f'{units.split()[-1]} Bins ({units})', title_fontsize=9)


def _empty_panel(ax, message):
    ax.text(0.5, 0.5, message, ha='center', va='center', transform=ax.transAxes,
            fontsize=12, color='#888')
    ax.axis('off')


if HAS_GPD and contiguous_states is not None:
    states_proj = contiguous_states.to_crs(epsg=5070).copy()

    # Resolve a state-code column on the shapefile side
    SHAPEFILE_STATE_CANDIDATES = ['STUSPS', 'STATE_ABBR', 'ABBR', 'STATEFP',
                                  'postal_code', 'STATE', 'state', 'STUSAB', 'stusps']
    shapefile_state_col = None
    for c in SHAPEFILE_STATE_CANDIDATES:
        if c in states_proj.columns:
            shapefile_state_col = c
            break

    if shapefile_state_col is None:
        print('!!  No state-code column on shapefile. states_proj columns:', list(states_proj.columns))
        state_geo = None
    elif STATE_COL is None:
        print('!!  STATE_COL on facility side was not resolved.')
        state_geo = None
    else:
        states_proj['_state_key'] = states_proj[shapefile_state_col].astype(str).str.strip().str.upper()
        if shapefile_state_col != STATE_COL:
            print(f'   shapefile state column = {shapefile_state_col!r}, '
                  f'facility state column = {STATE_COL!r} — bridging via _state_key')

        state_facets_loc = facility_ref.groupby(STATE_COL, as_index=False)[
            ['annual_energy_twh', 'annual_co2_mt']
        ].sum().copy()
        state_facets_loc['_state_key'] = (
            state_facets_loc[STATE_COL].astype(str).str.strip().str.upper()
        )

        state_geo = states_proj.merge(
            state_facets_loc.drop(columns=[STATE_COL]),
            on='_state_key', how='left',
        ).fillna({'annual_energy_twh': 0, 'annual_co2_mt': 0})
        n_with_data = (state_geo['annual_co2_mt'] > 0).sum()
        print(f'   joined HDC totals to {n_with_data} of {len(state_geo)} states.')

    if gdf_ba is not None and BA_GEOM_COL is not None:
        ba_facets = facility_ref.groupby(HDC_REGION_COL, as_index=False)[
            ['annual_energy_twh', 'annual_co2_mt']
        ].sum().rename(columns={HDC_REGION_COL: BA_GEOM_COL})
        ba_geo = gdf_ba.to_crs(epsg=5070).merge(ba_facets, on=BA_GEOM_COL, how='left').fillna(0)
    else:
        ba_geo = None

    fig, axes = plt.subplots(2, 2, figsize=(18, 12),
                             gridspec_kw={'wspace': 0.25, 'hspace': 0.05})

    fig.text(0.27, 0.93, 'Balancing Authorities', ha='center', fontsize=14, fontweight='bold')
    fig.text(0.73, 0.93, 'States', ha='center', fontsize=14, fontweight='bold')

    co2_palette = ['#fff7ec', '#fee8c8', '#fdd49e', '#fdbb84', '#fc8d59', '#d7301f']
    twh_palette = ['#f2f0f7', '#dadaeb', '#bcbddc', '#9e9ac8', '#756bb1', '#54278f']

    if ba_geo is not None:
        _plot_panel(axes[0, 0], ba_geo, co2_palette, 'A.', 'annual_co2_mt', 'MtCO₂')
    else:
        _empty_panel(axes[0, 0], 'BA shapefile missing')

    if state_geo is not None:
        _plot_panel(axes[0, 1], state_geo, co2_palette, 'B.', 'annual_co2_mt', 'MtCO₂')
    else:
        _empty_panel(axes[0, 1], 'state-data join failed (see warning above)')

    if ba_geo is not None:
        _plot_panel(axes[1, 0], ba_geo, twh_palette, 'C.', 'annual_energy_twh', 'TWh')
    else:
        _empty_panel(axes[1, 0], 'BA shapefile missing')

    if state_geo is not None:
        _plot_panel(axes[1, 1], state_geo, twh_palette, 'D.', 'annual_energy_twh', 'TWh')
    else:
        _empty_panel(axes[1, 1], 'state-data join failed (see warning above)')

    plt.savefig(FIGURE_DIR / 'figure_2_dc_maps.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_DIR / 'figure_2_dc_maps.png', dpi=200, bbox_inches='tight')
    plt.show()


## 18. Figure 3 — BA carbon-intensity choropleth

In [ ]:
# === Figure 3 — BA carbon-intensity choropleth (paper bins) ===
# Paper bins: < 369, 369-491, 491-551, 551-822, 822-985, 985-1016
PAPER_CI_EDGES = [-np.inf, 369, 491, 551, 822, 985, 1016, np.inf]
PAPER_CI_LABELS = [
    '< 369 gCO\u2082/kWh',
    '369–491 gCO\u2082/kWh',
    '491–551 gCO\u2082/kWh',
    '551–822 gCO\u2082/kWh',
    '822–985 gCO\u2082/kWh',
    '985–1016 gCO\u2082/kWh',
    '> 1016 gCO\u2082/kWh',
]
PAPER_CI_PALETTE = [
    '#cee5f2',  # < 369  light blue (below US avg)
    '#a3d597',  # 369-491 light green
    '#e3f0a8',  # 491-551 yellow-green
    '#f4a582',  # 551-822 light orange
    '#d62828',  # 822-985 red
    '#7b0d0d',  # 985-1016 dark red
    '#67000d',  # >1016 darkest
]


def _draw_ci_choropleth(geo_df, ci_col, base_layer=None, title_suffix=''):
    fig, ax = plt.subplots(figsize=(13, 8))
    if base_layer is not None:
        base_layer.plot(ax=ax, color='#f0f0f0', edgecolor='#888', linewidth=0.4)

    bins = pd.cut(geo_df[ci_col], bins=PAPER_CI_EDGES,
                  labels=PAPER_CI_LABELS, include_lowest=True)
    color_map = dict(zip(PAPER_CI_LABELS, PAPER_CI_PALETTE))
    geo_df = geo_df.copy()
    geo_df['_color'] = bins.astype(str).map(color_map).fillna('#d9d9d9')

    geo_df.plot(color=geo_df['_color'], edgecolor='#666', linewidth=0.4, ax=ax)
    ax.set_title(f'Carbon Intensity (gCO\u2082/kWh)\n(U.S. avg = 369 gCO\u2082/kWh){title_suffix}',
                 fontsize=12, fontweight='bold', loc='left')
    ax.axis('off')

    used_labels = [l for l in PAPER_CI_LABELS if l in bins.astype(str).values]
    legend_elements = [Patch(facecolor=color_map[l], edgecolor='black', label=l)
                       for l in used_labels]
    legend_elements.append(Patch(facecolor='#d9d9d9', edgecolor='black', label='No data'))
    ax.legend(handles=legend_elements, loc='center left',
              bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=10,
              title='Carbon Intensity', title_fontsize=11)
    plt.tight_layout()
    return fig


# Try BA-level first, fall back to state-level if BA shapefile absent
if HAS_GPD and gdf_ba is not None and BA_GEOM_COL is not None:
    ci_geo = gdf_ba.to_crs(epsg=5070).merge(
        ba_ef.rename(columns={'BACODE': BA_GEOM_COL})
             [[BA_GEOM_COL, 'ci_combustion_g_per_kwh']],
        on=BA_GEOM_COL, how='left',
    )
    fig = _draw_ci_choropleth(ci_geo, 'ci_combustion_g_per_kwh',
                              base_layer=contiguous_states.to_crs(epsg=5070))
    fig.savefig(FIGURE_DIR / 'figure_3_carbon_intensity.pdf', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'figure_3_carbon_intensity.png', dpi=200, bbox_inches='tight')
    plt.show()
elif HAS_GPD and contiguous_states is not None:
    # State-level fallback: load-weighted avg CI per state
    if STATE_COL is not None:
        state_ci = (
            facility_ref.groupby(STATE_COL).apply(
                lambda d: (d['annual_co2_mt'].sum() / d['annual_energy_twh'].sum()) * 1000.0
                if d['annual_energy_twh'].sum() > 0 else np.nan
            ).reset_index()
        )
        state_ci.columns = [STATE_COL, 'ci_g_per_kwh']
        ci_geo = contiguous_states.to_crs(epsg=5070).merge(
            state_ci, on=STATE_COL, how='left'
        )
        fig = _draw_ci_choropleth(
            ci_geo, 'ci_g_per_kwh',
            title_suffix='\n(state-level fallback — BA shapefile not provided)'
        )
        fig.savefig(FIGURE_DIR / 'figure_3_carbon_intensity.pdf', dpi=300, bbox_inches='tight')
        fig.savefig(FIGURE_DIR / 'figure_3_carbon_intensity.png', dpi=200, bbox_inches='tight')
        plt.show()
        print('Figure 3 drawn at state level. For BA-level (paper-faithful), add '
              "balancing_authorities_polygons/balancing_authorities_EPA.shp.")
    else:
        print('Figure 3 skipped — no state column available for fallback.')
else:
    print('Figure 3 skipped — no map data.')


## 19. Figure 4 — fuel-mix bar chart for top 7 BAs and US Total

In [ ]:
# === Figure 4 — fuel-mix bar chart, ordered by HDC TWh demand ===
FUEL_COLORS = {
    'COAL':       '#8B1A1A',
    'GAS':        '#F28E2B',
    'OIL':        '#E15759',
    'OFSL':       '#76B7B2',  # paper has teal for OFSL
    'OTHF':       '#59A14F',
    'NUCLEAR':    '#F1C232',
    'BIOMASS':    '#1A9850',
    'GEOTHERMAL': '#66BD63',
    'HYDRO':      '#3288BD',
    'SOLAR':      '#FDBF6F',
    'WIND':       '#85C1A8',  # light teal-green to match paper
}

# top 7 BAs by HDC TWh demand
top_bas = ba_summary.head(7)[HDC_REGION_COL].tolist()
top_share = ba_fuel_share.reindex(top_bas).fillna(0)

# National HDC-load-weighted row
national_share = (
    ba_fuel_share.reindex(hdc_load_per_ba.index).fillna(0)
                 .mul(hdc_load_per_ba['hdc_twh'], axis=0)
                 .sum(axis=0) / hdc_load_per_ba['hdc_twh'].sum()
)
national_row = pd.DataFrame([national_share], index=['US Total'])
plot_data = pd.concat([national_row, top_share], axis=0)
plot_data = plot_data[FUEL_ORDER]

# TWh labels, indexed in same order as plot_data
twh_labels = pd.concat([
    pd.Series({'US Total': float(facility_ref['annual_energy_twh'].sum())}),
    ba_summary.set_index(HDC_REGION_COL).loc[top_bas, 'electricity_twh'],
])

# *** PAPER ORDER: US Total at TOP, then BAs ranked by HDC TWh demand DESCENDING ***
# matplotlib's barh draws index[0] at the bottom, so reverse so US Total ends up on top
plot_data = plot_data.iloc[::-1]
twh_labels = twh_labels.reindex(plot_data.index)

group_shares = pd.DataFrame(index=plot_data.index)
group_shares['Fossil']     = plot_data[[c for c in FOSSIL    if c in plot_data.columns]].sum(axis=1)
group_shares['Nuclear']    = plot_data[[c for c in NUCLEAR_S if c in plot_data.columns]].sum(axis=1)
group_shares['Renewables'] = plot_data[[c for c in RENEWABLE if c in plot_data.columns]].sum(axis=1)

fig, ax = plt.subplots(figsize=(15, 9))
plot_data.plot(
    kind='barh', stacked=True,
    color=[FUEL_COLORS[f] for f in plot_data.columns],
    ax=ax, edgecolor='none', linewidth=0,
    width=0.7,
)

# TWh labels on the right
for i, region in enumerate(plot_data.index):
    ax.text(1.02, i, f'{twh_labels[region]:0.2f} TWh',
            va='center', ha='left', fontsize=11, fontweight='bold',
            transform=ax.get_yaxis_transform())

# Inline % labels (slices ≥ 4%)
for i, region in enumerate(plot_data.index):
    cumulative = 0.0
    for fuel in plot_data.columns:
        v = plot_data.loc[region, fuel]
        pct = v * 100
        if pct >= 4:
            ax.text(cumulative + v / 2, i, f'{pct:.1f}%',
                    ha='center', va='center', fontsize=9,
                    color='white', fontweight='bold')
        cumulative += v

# Group summary above each bar
for i, region in enumerate(plot_data.index):
    s = (f"Fossil {group_shares.loc[region, 'Fossil']*100:.1f}%   |   "
         f"Nuclear {group_shares.loc[region, 'Nuclear']*100:.1f}%   |   "
         f"Renewables {group_shares.loc[region, 'Renewables']*100:.1f}%")
    ax.text(0.5, i + 0.36, s, ha='center', va='bottom', fontsize=10)

ax.set_xlim(0, 1)
ax.set_xlabel('Fuel mix share', labelpad=8, fontsize=12, fontweight='bold')
ax.set_ylabel('US and balancing authority', labelpad=8, fontsize=12, fontweight='bold')
ax.set_yticklabels(plot_data.index, fontweight='bold')

# Custom grouped legend with section headers
fossil_share    = float(national_share.reindex(list(FOSSIL)).fillna(0).sum())
nuclear_share   = float(national_share.reindex(list(NUCLEAR_S)).fillna(0).sum())
renewable_share = float(national_share.reindex(list(RENEWABLE)).fillna(0).sum())

handles = []
labels  = []

handles.append(Line2D([], [], linestyle='none'))
labels.append(f'Fossil fuels ({fossil_share*100:.1f}%)')
for f in ['COAL', 'GAS', 'OIL', 'OFSL', 'OTHF']:
    if f in FUEL_COLORS:
        handles.append(Line2D([0], [0], marker='s', color='none',
                              markerfacecolor=FUEL_COLORS[f], markersize=10))
        labels.append(f'   {f}')

handles.append(Line2D([], [], linestyle='none'))
labels.append(f'Nuclear ({nuclear_share*100:.1f}%)')
handles.append(Line2D([0], [0], marker='s', color='none',
                      markerfacecolor=FUEL_COLORS['NUCLEAR'], markersize=10))
labels.append('   NUCLEAR')

handles.append(Line2D([], [], linestyle='none'))
labels.append(f'Renewables ({renewable_share*100:.1f}%)')
for f in ['BIOMASS', 'GEOTHERMAL', 'HYDRO', 'SOLAR', 'WIND']:
    if f in FUEL_COLORS:
        handles.append(Line2D([0], [0], marker='s', color='none',
                              markerfacecolor=FUEL_COLORS[f], markersize=10))
        labels.append(f'   {f}')

ax.legend(handles, labels, loc='center left', bbox_to_anchor=(1.18, 0.5),
          frameon=False, title='Fuel groups & types', title_fontsize=11,
          fontsize=9, handlelength=1.2)

for spine in ('top', 'right', 'left', 'bottom'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'figure_4_fuel_mix.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'figure_4_fuel_mix.png', dpi=200, bbox_inches='tight')
plt.show()


## 20. Print all numbers for paper update

Every number that appears in the paper, all in one place. Copy-paste the
output of the cell below into your draft.


In [ ]:
def section(title):
    print('\n' + '=' * 78)
    print(f'  {title}')
    print('=' * 78)


# Helpful aliases
total_twh_ref = facility_ref['annual_energy_twh'].sum()
total_mt_ref  = facility_ref['annual_co2_mt'].sum()
ci_ref        = (total_mt_ref / total_twh_ref) * 1000.0
total_twh_lo  = scenario_facilities['low_load']['annual_energy_twh'].sum()
total_mt_lo   = scenario_facilities['low_load']['annual_co2_mt'].sum()
total_twh_mid = scenario_facilities['intermediate']['annual_energy_twh'].sum()
total_mt_mid  = scenario_facilities['intermediate']['annual_co2_mt'].sum()


section(f'1.  HEADLINE NUMBERS  (eGRID {EGRID_YEAR})')
n_dc = len(facility_ref)
print(f'  Number of HDCs analysed                       : {n_dc}')
print(f'  Total electricity (low load,  u=0.48)  TWh    : {total_twh_lo:0.2f}')
print(f'  Total electricity (intermediate u=0.58) TWh   : {total_twh_mid:0.2f}')
print(f'  Total electricity (reference, u=0.663) TWh    : {total_twh_ref:0.2f}')
print(f'  Total CO\u2082-eq (low load,  u=0.48)   MT          : {total_mt_lo:0.2f}')
print(f'  Total CO\u2082-eq (intermediate u=0.58) MT          : {total_mt_mid:0.2f}')
print(f'  Total CO\u2082-eq (reference, u=0.663) MT           : {total_mt_ref:0.2f}')
print(f'  Weighted-avg carbon intensity (g/kWh)         : {ci_ref:0.1f}')
print(f'  US national reference CI (g/kWh)              : {US_NATIONAL_CI:0.0f}')
print(f'  HDCs above national CI threshold (% of TWh)   : {share*100:0.1f}%')


section('2.  HDC FACILITY CHARACTERISTICS (paper §Characteristics …)')
mw = pd.to_numeric(dc[CAPACITY_COL], errors='coerce')
print(f'  Number of HDCs                                : {n_dc}')
print(f'  Mean facility power capacity (MW)             : {mw.mean():0.1f}')
print(f'  Median facility power capacity (MW)           : {mw.median():0.1f}')
if SQFT_COL is not None and SQFT_COL in dc.columns:
    SQFT_TO_M2 = 0.092903
    sqft = pd.to_numeric(dc[SQFT_COL], errors='coerce')
    valid = sqft.notna() & (sqft > 0) & mw.notna() & (mw > 0)
    density = (mw[valid] * 1e6) / (sqft[valid] * SQFT_TO_M2)
    z = (density - density.mean()) / density.std()
    trimmed = density[z.abs() <= 2]
    print(f'  Mean power density   (W/m\u00b2, raw)              : {density.mean():,.0f}')
    print(f'  Mean power density   (W/m\u00b2, |Z|<=2)           : {trimmed.mean():,.0f}')
    print(f'  Median power density (W/m\u00b2, |Z|<=2)           : {trimmed.median():,.0f}')


section('3.  TOP-10 STATES (paper Table 1, reference scenario)')
if state_summary is not None:
    table1 = state_summary.head(10).copy()
    cols_show = [STATE_COL, 'co2_mt', 'electricity_twh', 'n_facilities',
                 'mean_electricity_per_dc_twh', 'mean_co2_per_dc_mt',
                 'carbon_intensity_g_per_kwh']
    rename = {
        STATE_COL: 'state',
        'co2_mt': 'CO\u2082 (MT)',
        'electricity_twh': 'TWh',
        'n_facilities': 'n HDCs',
        'mean_electricity_per_dc_twh': 'mean TWh/DC',
        'mean_co2_per_dc_mt': 'mean MT/DC',
        'carbon_intensity_g_per_kwh': 'CI (g/kWh)',
    }
    if SQFT_COL and SQFT_COL in facility_ref.columns:
        # mean facility size, in thousand m^2 (paper convention)
        SQFT_TO_M2 = 0.092903
        msize = facility_ref.groupby(STATE_COL)[SQFT_COL].mean() * SQFT_TO_M2 / 1000.0
        table1 = table1.merge(msize.rename('mean_size_km2').reset_index(), on=STATE_COL, how='left')
        cols_show.append('mean_size_km2')
        rename['mean_size_km2'] = 'mean size (1000 m\u00b2)'
    table1 = table1[cols_show].rename(columns=rename)
    print(table1.round({'CO\u2082 (MT)': 2, 'TWh': 2, 'mean TWh/DC': 2,
                       'mean MT/DC': 2, 'CI (g/kWh)': 0,
                       'mean size (1000 m\u00b2)': 0}).to_string(index=False))


section('4.  BA-LEVEL SUMMARY (paper §Carbon Intensity / Fig. 3 / Fig. 4)')
ba_show = ba_summary.head(15).copy()
ba_show = ba_show[[HDC_REGION_COL, 'n_facilities', 'electricity_twh', 'co2_mt',
                   'ci_g_per_kwh', 'total_capacity_mw']]
ba_show.columns = ['BA', 'n HDCs', 'TWh', 'CO\u2082 (MT)', 'CI (g/kWh)', 'capacity (MW)']
print(ba_show.round({'TWh': 2, 'CO\u2082 (MT)': 2, 'CI (g/kWh)': 0, 'capacity (MW)': 0})
       .to_string(index=False))


section('5.  PER-BA FUEL MIX (paper Fig. 4)')
# Same top 7 BAs the paper uses, plus US Total
top7 = ba_summary.head(7)[HDC_REGION_COL].tolist()
fuel_table = ba_fuel_share.reindex(top7).copy() * 100.0
# add national row
nat = (
    ba_fuel_share.reindex(hdc_load_per_ba.index).fillna(0)
                 .mul(hdc_load_per_ba['hdc_twh'], axis=0)
                 .sum(axis=0) / hdc_load_per_ba['hdc_twh'].sum()
) * 100.0
fuel_table.loc['US Total'] = nat
# group columns
fuel_table['Fossil']     = fuel_table[[c for c in FOSSIL    if c in fuel_table.columns]].sum(axis=1)
fuel_table['Nuclear']    = fuel_table[[c for c in NUCLEAR_S if c in fuel_table.columns]].sum(axis=1)
fuel_table['Renewables'] = fuel_table[[c for c in RENEWABLE if c in fuel_table.columns]].sum(axis=1)
# put US Total first
fuel_table = pd.concat([fuel_table.loc[['US Total']], fuel_table.drop('US Total')])
print(fuel_table.round(1).to_string())


section('6.  NATIONAL FUEL MIX, ATTRIBUTED (paper §Results, Fig. 4)')
print('  Fuel              Attributed TWh    Share (%)')
print('  ' + '-' * 50)
for fuel in national_fuel_table.sort_values('share_pct', ascending=False).index:
    twh = float(national_fuel_table.loc[fuel, 'attributed_twh'])
    pct = float(national_fuel_table.loc[fuel, 'share_pct'])
    print(f'  {fuel:18}{twh:12.2f}    {pct:7.2f}')
print()
print('  Fuel-group totals:')
for grp in ['fossil', 'nuclear', 'renewable']:
    if grp in group_table.index:
        twh = float(group_table.loc[grp, 'attributed_twh'])
        pct = float(group_table.loc[grp, 'share_pct'])
        paper = group_table.loc[grp, 'paper_pct'] if 'paper_pct' in group_table.columns else None
        line = f'  {grp:12}        {twh:8.2f} TWh    {pct:6.2f}%'
        if paper is not None and not pd.isna(paper):
            line += f'    (paper: {paper:.1f}%)'
        print(line)


section('7.  THREE-SCENARIO TOTALS (paper Table S.3.1)')
print(scenarios.round(2).to_string())


section('8.  KEY PAPER CLAIMS — DOES THIS NOTEBOOK REPRODUCE THEM?')
claims = [
    ('We identified 403 HDCs', 403, n_dc),
    ('Consumed 67.82 TWh (low-load)', 67.82, total_twh_lo),
    ('Consumed 81.94 TWh (intermediate)', 81.94, total_twh_mid),
    ('Consumed 93.66 TWh (reference)', 93.66, total_twh_ref),
    ('38.14 MT CO\u2082-eq (low-load)', 38.14, total_mt_lo),
    ('46.09 MT CO\u2082-eq (intermediate)', 46.09, total_mt_mid),
    ('52.69 MT CO\u2082-eq (reference)', 52.69, total_mt_ref),
    ('564 g/kWh weighted-avg CI', 564.0, ci_ref),
    ('~96% of HDCs above national avg', 96.0, share * 100),
    ('Fossil 56.3% of attributed', 56.3,
        float(group_table.loc['fossil', 'share_pct']) if 'fossil' in group_table.index else np.nan),
    ('Nuclear 20.0% of attributed', 20.0,
        float(group_table.loc['nuclear', 'share_pct']) if 'nuclear' in group_table.index else np.nan),
    ('Renewable 23.7% of attributed', 23.7,
        float(group_table.loc['renewable', 'share_pct']) if 'renewable' in group_table.index else np.nan),
]
print(f'  {"Claim":50} {"Paper":>10} {"This run":>10} {"Δ %":>8}')
print('  ' + '-' * 80)
for label, paper_v, ours in claims:
    if pd.isna(ours):
        delta_pct = float('nan')
        line = f'  {label:50} {paper_v:>10.2f} {"N/A":>10} {"N/A":>8}'
    else:
        delta_pct = (ours - paper_v) / paper_v * 100.0
        line = f'  {label:50} {paper_v:>10.2f} {ours:>10.2f} {delta_pct:>7.2f}%'
    print(line)


## 21. Export results

In [ ]:
facility_ref.to_csv(OUTPUT_DIR / 'facility_reference.csv', index=False)
ba_ef.to_csv(OUTPUT_DIR / 'ba_effective_emission_factor.csv', index=False)
ba_summary.to_csv(OUTPUT_DIR / 'ba_summary.csv', index=False)
ba_fuel_share.to_csv(OUTPUT_DIR / 'ba_fuel_share.csv')
ba_grouped.to_csv(OUTPUT_DIR / 'ba_fuel_groups.csv')
diag.to_csv(OUTPUT_DIR / 'paper_alignment_fig4.csv')
scenarios.to_csv(OUTPUT_DIR / 'scenarios.csv')
national_fuel_table.to_csv(OUTPUT_DIR / 'national_fuel_mix.csv')
group_table.to_csv(OUTPUT_DIR / 'national_fuel_groups.csv')
alignment.to_csv(OUTPUT_DIR / 'paper_alignment.csv', index=False)
if state_summary is not None:
    state_summary.to_csv(OUTPUT_DIR / 'state_summary.csv', index=False)

print(f'Wrote tabular outputs to {OUTPUT_DIR}:')
for p in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  - {p.name}')
print()
print(f'Wrote figures to {FIGURE_DIR}:')
for p in sorted(FIGURE_DIR.glob('*')):
    print(f'  - {p.name}')
